# POO — Exercice 3 : la classe `Instant` (méthodes statiques)

**Notions travaillées :** méthodes **statiques** (`@staticmethod`), validation de données, méthodes spéciales de comparaison.

> **Les méthodes statiques (hors programme)**
> Une méthode classique travaille sur **un objet** (`self`). Une méthode **statique** ne travaille sur aucun objet : elle n'a pas de `self` et on l'appelle directement depuis la classe, par exemple `Instant.est_bissextile(2024)`.
> On l'utilise pour une fonction **liée au thème de la classe** mais qui ne dépend d'aucun objet particulier : savoir si 2024 est bissextile ne dépend d'aucun instant précis.

⚠️ Le fichier `instant.py` mentionné dans l'énoncé n'est **pas fourni**. On propose donc une version complète et cohérente avec l'énoncé (format `jj/mm/aaaa:hh:mm:ss`, méthodes `est_valide` et `est_bissextile`). Si le fichier distribué contient d'autres méthodes, la démarche reste la même.

## Les méthodes statiques

**Année bissextile** (règle du calendrier grégorien) : une année est bissextile si elle est divisible par 4 **mais pas** par 100, **ou** si elle est divisible par 400.
Exemples : 2024 oui, 1900 non (divisible par 100 mais pas par 400), 2000 oui (divisible par 400).

**Nombre de jours d'un mois** : on range les durées dans une liste (indice 0 = janvier). Février dépend de l'année, donc on appelle `est_bissextile`.

In [1]:
class Instant:

    # ---------- méthodes statiques : pas de self ----------
    @staticmethod
    def est_bissextile(annee):
        return (annee % 4 == 0 and annee % 100 != 0) or annee % 400 == 0

    @staticmethod
    def jours_dans_mois(mois, annee):
        jours = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
        if mois == 2 and Instant.est_bissextile(annee):
            return 29
        return jours[mois - 1]

    # ---------- constructeur ----------
    def __init__(self, texte):
        # texte au format "jj/mm/aaaa:hh:mm:ss"
        date, h, mi, s = texte.split(":")
        j, mo, a = date.split("/")
        self.jour, self.mois, self.annee = int(j), int(mo), int(a)
        self.heure, self.minute, self.seconde = int(h), int(mi), int(s)

    # ---------- méthodes classiques : avec self ----------
    def est_valide(self):
        if not 1 <= self.mois <= 12:
            return False
        if not 1 <= self.jour <= Instant.jours_dans_mois(self.mois, self.annee):
            return False
        return 0 <= self.heure < 24 and 0 <= self.minute < 60 and 0 <= self.seconde < 60

    def __str__(self):
        # :02d -> sur 2 chiffres, complété par des zéros (5 -> "05")
        return f"{self.jour:02d}/{self.mois:02d}/{self.annee:04d}:{self.heure:02d}:{self.minute:02d}:{self.seconde:02d}"

    def en_tuple(self):
        # ordre du plus important au moins important : permet de comparer facilement
        return (self.annee, self.mois, self.jour, self.heure, self.minute, self.seconde)

    def __eq__(self, autre):
        return self.en_tuple() == autre.en_tuple()

    def __lt__(self, autre):
        return self.en_tuple() < autre.en_tuple()

    def ajoute_secondes(self, n):
        # renvoie un NOUVEL instant, n secondes plus tard
        r = Instant(str(self))
        r.seconde += n
        r.minute += r.seconde // 60
        r.seconde %= 60
        r.heure += r.minute // 60
        r.minute %= 60
        r.jour += r.heure // 24
        r.heure %= 24
        while r.jour > Instant.jours_dans_mois(r.mois, r.annee):
            r.jour -= Instant.jours_dans_mois(r.mois, r.annee)
            r.mois += 1
            if r.mois > 12:
                r.mois = 1
                r.annee += 1
        return r

## Explications

- **`@staticmethod`** se place juste au-dessus de la méthode. Dans la classe, on l'appelle avec le **nom de la classe** : `Instant.est_bissextile(self.annee)`.
- **Constructeur :** `split(":")` découpe `"14/07/2026:09:05:00"` en `["14/07/2026", "09", "05", "00"]`, puis `split("/")` découpe la date. On convertit chaque morceau avec `int`.
- **`est_valide`** vérifie chaque champ. Le jour maximal dépend du mois **et** de l'année (29 février), d'où l'appel à la méthode statique.
- **Comparer deux instants :** Python compare les tuples **élément par élément**, de gauche à droite (ordre lexicographique). En plaçant l'année en premier, puis le mois, etc., `<` sur les tuples donne exactement l'ordre chronologique.
- **`ajoute_secondes`** fonctionne comme une **addition avec retenue** : les secondes en trop font des minutes (`// 60` pour la retenue, `% 60` pour le reste), et ainsi de suite jusqu'aux jours, puis aux mois et aux années.

In [2]:
# Méthode statique : appelée sur la classe, sans objet
print(Instant.est_bissextile(2024), Instant.est_bissextile(1900), Instant.est_bissextile(2000))

t = Instant("31/12/2026:23:59:30")
print(t, "valide :", t.est_valide())
print("+ 45 s ->", t.ajoute_secondes(45))          # passage à l'année suivante

print(Instant("29/02/2025:10:00:00").est_valide())   # False : 2025 n'est pas bissextile
print(Instant("29/02/2024:10:00:00").est_valide())   # True
print(Instant("14/07/2026:25:00:00").est_valide())   # False : 25 h

a = Instant("14/07/2026:09:05:00")
b = Instant("01/08/2026:08:00:00")
print(a < b, a == Instant("14/07/2026:09:05:00"))

assert str(Instant("1/2/2026:3:4:5")) == "01/02/2026:03:04:05"
assert str(Instant("28/02/2024:23:00:00").ajoute_secondes(3600)) == "29/02/2024:00:00:00"

True False True
31/12/2026:23:59:30 valide : True
+ 45 s -> 01/01/2027:00:00:15
False
True
False
True True


💡 **À retenir**
- Méthode **classique** : elle a besoin d'un objet → `self`, appel `t.est_valide()`.
- Méthode **statique** : elle n'a besoin d'aucun objet → pas de `self`, appel `Instant.est_bissextile(2024)`.